# MNIST Vision Transformer on 2x T4 (PyTorch counterpart)

This notebook runs the PyTorch counterpart of the hand-written CUDA ViT. It is
set up for comparison against `kaggle_run.ipynb`:

- same Kaggle CSV search and torchvision MNIST fallback;
- same model config: V=256, T=784, L=2, D=64, H=4, C=10;
- same train runs: Adam lr=1e-3, 3000 steps, per-rank B=32;
- same log columns and throughput timer boundaries.

The PyTorch script is written in the normal ML style: `nn.Module` layers,
`DataLoader`, `torch.optim.Adam`, and `DistributedDataParallel` for two GPUs.
The architecture and experiment settings stay aligned with the CUDA notebook.
Attention is pinned to PyTorch's SDPA math backend in the source so the run
does not depend on automatic Flash or memory-efficient backend selection.

For throughput, use the same Kaggle accelerator and the same CSV in both
notebooks. Loss/accuracy curves are a training-behavior comparison, not an
exact numeric parity test: C++ and PyTorch RNG streams do not give identical
initial weights or sampled batches from seed 42.

Kaggle setup:

1. Select `GPU T4 x2` in Settings.
2. Enable Internet (needed to clone the repo; also for the MNIST fallback download).
3. Add the Digit Recognizer dataset when comparing with the CUDA notebook.


In [ ]:
!rm -rf /tmp/hpc && git clone https://github.com/SadreevAmir/hpc_final_project /tmp/hpc && cp /tmp/hpc/src/train_vit_torch.py .
!ls -lh train_vit_torch.py


## 1. Verify the environment

Expect two T4 GPUs and a CUDA PyTorch build.

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total --format=csv
import torch
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
print('cuda devices:', torch.cuda.device_count())
assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator'


## 2. Check the model contract

The default PyTorch model should keep the CUDA-comparable parameter count.

In [ ]:
from train_vit_torch import Cfg, DEFAULT_PARAM_COUNT, PixelViT, count_parameters
cfg = Cfg()
model = PixelViT(cfg)
print(cfg)
print('parameter count:', count_parameters(model))
assert count_parameters(model) == DEFAULT_PARAM_COUNT


## 3. Locate or materialize the data

This is the same CSV search order and MNIST fallback as the CUDA notebook.

In [ ]:
import glob
import os
import numpy as np

known = [
    '/kaggle/input/digit-recognizer/train.csv',
    '/kaggle/input/fashionmnist/fashion-mnist_train.csv',
    '/kaggle/input/fashion-mnist/fashion-mnist_train.csv',
]
globbed = sorted(set(
    glob.glob('/kaggle/input/**/*train*.csv', recursive=True) +
    glob.glob('/kaggle/input/**/*Train*.csv', recursive=True)))

def valid_csv(path):
    try:
        with open(path) as f:
            return len(f.readline().split(',')) == 785
    except Exception:
        return False

CSV = next((c for c in known + globbed if os.path.exists(c) and valid_csv(c)), None)

if CSV is None:
    print('Falling back to torchvision MNIST...')
    from torchvision.datasets import MNIST
    ds = MNIST(root='/kaggle/working/mnist_raw', train=True, download=True)
    labels = ds.targets.numpy().astype(np.int32)
    pixels = ds.data.numpy().reshape(-1, 784).astype(np.int32)
    arr = np.concatenate([labels[:, None], pixels], axis=1)
    header = 'label,' + ','.join(f'pixel{i}' for i in range(784))
    CSV = '/kaggle/working/train.csv'
    np.savetxt(CSV, arr, fmt='%d', delimiter=',', header=header, comments='')

assert os.path.exists(CSV), CSV
print('Using CSV:', CSV)
print('Size     :', os.path.getsize(CSV) // (1024 * 1024), 'MiB')
os.environ['CSV'] = CSV
!head -c 120 "$CSV" ; echo
!wc -l "$CSV"


## 4. Training smoke test

Run one normal PyTorch step before profiling. This leaves the real training traceback visible if the embedded source or CUDA environment is stale.

In [ ]:
!python train_vit_torch.py "$CSV" 1 8 0.001 \
  --device cuda --log-path torch_smoke_log.csv


## 5. GPU, memory, and DDP profiling

Runs 50 PyTorch steps for B in {8, 16, 32, 64} on one GPU, plus one two-GPU run at B=32. The internal throughput timer matches the training script timer and excludes model/data setup.

In [ ]:
import os
import subprocess
import threading
import time
import numpy as np

try:
    import psutil
except ImportError:
    subprocess.run(['pip', 'install', '-q', 'psutil'], check=True)
    import psutil

def _gpu_monitor(stop_evt, records, interval=0.25):
    while not stop_evt.is_set():
        r = subprocess.run(
            ['nvidia-smi',
             '--query-gpu=index,utilization.gpu,memory.used,memory.total,power.draw',
             '--format=csv,noheader,nounits'],
            capture_output=True, text=True)
        ts = time.time()
        for line in r.stdout.strip().splitlines():
            parts = [x.strip() for x in line.split(',')]
            if len(parts) < 5:
                continue
            try:
                records.append(dict(
                    ts=ts, gpu=int(parts[0]),
                    gpu_util=float(parts[1]),
                    mem_mb=float(parts[2]),
                    mem_total=float(parts[3]),
                    power=float(parts[4]) if 'N/A' not in parts[4] else 0.0,
                ))
            except ValueError:
                pass
        time.sleep(interval)

def run_prof(csv_path, B, steps=50, lr=0.001, np_=1):
    if np_ == 1:
        cmd = ['python', 'train_vit_torch.py', csv_path, str(steps), str(B), str(lr),
               '--device', 'cuda', '--log-path', 'torch_profile_log.csv']
    else:
        cmd = ['torchrun', '--standalone', f'--nproc_per_node={np_}',
               '--master_port=29531', 'train_vit_torch.py',
               csv_path, str(steps), str(B), str(lr),
               '--device', 'cuda', '--log-path', 'torch_profile_log.csv']
    stop = threading.Event()
    recs, cpu_s = [], []

    def _cpu():
        while not stop.is_set():
            cpu_s.append((time.time(), psutil.cpu_percent()))
            time.sleep(0.25)

    gmon = threading.Thread(target=_gpu_monitor, args=(stop, recs), daemon=True)
    cmon = threading.Thread(target=_cpu, daemon=True)
    gmon.start()
    cmon.start()

    t0 = time.time()
    proc = subprocess.run(cmd, capture_output=True, text=True)
    elapsed = time.time() - t0

    stop.set()
    time.sleep(0.4)
    if proc.returncode:
        raise RuntimeError(
            f"profile run failed: {cmd}\n"
            f"--- stdout ---\n{proc.stdout}\n"
            f"--- stderr ---\n{proc.stderr}"
        )

    tput = None
    for line in proc.stdout.splitlines():
        if 'img/s' in line and 'throughput' in line.lower():
            for tok in line.replace('|', ' ').split():
                try:
                    tput = float(tok)
                    break
                except ValueError:
                    pass
            break

    return dict(recs=recs, cpu=cpu_s, elapsed=elapsed,
                tput=tput, stdout=proc.stdout)

CSV = os.environ.get('CSV', '')
assert CSV, 'Run the data cell first'
n_gpus = int(subprocess.check_output(
    'nvidia-smi --query-gpu=name --format=csv,noheader | wc -l',
    shell=True, text=True).strip())
print(f'GPUs detected: {n_gpus}')

prof = {}
BATCH_SIZES = [8, 16, 32, 64]
for B in BATCH_SIZES:
    print(f'  1-GPU  B={B:3d} / 50 steps ...', end=' ', flush=True)
    prof[('1gpu', B)] = run_prof(CSV, B, steps=50, np_=1)
    print(f"done  {prof[('1gpu', B)]['tput']} img/s  "
          f"elapsed={prof[('1gpu', B)]['elapsed']:.1f}s")

if n_gpus >= 2:
    print('  2-GPU  B= 32 / 50 steps (PyTorch DDP) ...',
          end=' ', flush=True)
    prof[('2gpu', 32)] = run_prof(CSV, 32, steps=50, np_=2)
    print(f"done  {prof[('2gpu', 32)]['tput']} img/s  "
          f"elapsed={prof[('2gpu', 32)]['elapsed']:.1f}s")
else:
    print('Single-GPU session - 2-GPU profiling skipped.')


## 5b. Plot profiling results

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

BATCH_SIZES = [8, 16, 32, 64]
cmap = plt.cm.tab10(np.linspace(0, 0.8, len(BATCH_SIZES)))

def gpu_ts(recs, gpu_idx=0):
    rows = [r for r in recs if r['gpu'] == gpu_idx]
    if not rows:
        return [], [], [], []
    t0 = rows[0]['ts']
    return (
        [r['ts'] - t0 for r in rows],
        [r['gpu_util'] for r in rows],
        [r['mem_mb'] / 1024 for r in rows],
        [r['power'] for r in rows],
    )

fig1, axes = plt.subplots(4, len(BATCH_SIZES), figsize=(5 * len(BATCH_SIZES), 12))
fig1.suptitle('PyTorch GPU and CPU time-series - 50 steps, 1 GPU',
              fontsize=13, fontweight='bold')

for idx, (B, color) in enumerate(zip(BATCH_SIZES, cmap)):
    exp = prof.get(('1gpu', B))
    if not exp:
        for row in range(4):
            axes[row][idx].set_visible(False)
        continue
    ts, util, mem, pwr = gpu_ts(exp['recs'], 0)
    for row, (values, label) in enumerate([
        (util, 'GPU util %'),
        (mem, 'GPU mem GiB'),
        (pwr, 'GPU power W'),
    ]):
        ax = axes[row][idx]
        ax.plot(ts, values, color=color, linewidth=1.3)
        ax.fill_between(ts, values, alpha=0.12, color=color)
        ax.set_title(f'B={B}', fontsize=10)
        ax.set_xlabel('Time (s)', fontsize=8)
        ax.set_ylabel(label, fontsize=8)
        ax.grid(alpha=0.3)
        ax.set_ylim(bottom=0)

    ax_cpu = axes[3][idx]
    if exp['cpu']:
        ct0 = exp['cpu'][0][0]
        ax_cpu.plot([x[0] - ct0 for x in exp['cpu']],
                    [x[1] for x in exp['cpu']], color=color, linewidth=1.3)
    ax_cpu.set_title(f'B={B}', fontsize=10)
    ax_cpu.set_xlabel('Time (s)', fontsize=8)
    ax_cpu.set_ylabel('CPU util %', fontsize=8)
    ax_cpu.grid(alpha=0.3)
    ax_cpu.set_ylim(0, 105)

plt.tight_layout()
plt.savefig('torch_profiling_timeseries.png', dpi=130, bbox_inches='tight')
plt.show()

fig2, axes2 = plt.subplots(2, 2, figsize=(12, 8))
fig2.suptitle('PyTorch summary metrics vs batch size - 50 steps, 1 GPU',
              fontsize=13, fontweight='bold')

tputs = [prof.get(('1gpu', B), {}).get('tput') or 0 for B in BATCH_SIZES]
peak_mem = [max((r['mem_mb'] for r in prof.get(('1gpu', B), {}).get('recs', [])
                 if r['gpu'] == 0), default=0) / 1024 for B in BATCH_SIZES]
avg_util, avg_cpu = [], []
for B in BATCH_SIZES:
    recs = prof.get(('1gpu', B), {}).get('recs', [])
    cpu = prof.get(('1gpu', B), {}).get('cpu', [])
    gpu_values = [r['gpu_util'] for r in recs if r['gpu'] == 0]
    avg_util.append(np.mean(gpu_values) if gpu_values else 0)
    avg_cpu.append(np.mean([v for _, v in cpu]) if cpu else 0)

labels = [f'B={B}' for B in BATCH_SIZES]
for ax, values, ylabel, title in [
    (axes2[0, 0], tputs, 'img/s', 'Throughput'),
    (axes2[0, 1], peak_mem, 'GiB', 'Peak GPU memory'),
    (axes2[1, 0], avg_util, '%', 'Average GPU utilization'),
    (axes2[1, 1], avg_cpu, '%', 'Average CPU utilization'),
]:
    bars = ax.bar(labels, values, color=cmap, edgecolor='white', linewidth=0.5)
    ax.bar_label(bars, fmt='%.1f', fontsize=9, padding=2)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('torch_profiling_summary.png', dpi=130, bbox_inches='tight')
plt.show()

if ('2gpu', 32) in prof:
    fig3, (ax_u, ax_m) = plt.subplots(1, 2, figsize=(14, 5))
    fig3.suptitle('PyTorch 1 GPU vs 2 GPU at B=32', fontsize=13, fontweight='bold')
    ts1, u1, m1, _ = gpu_ts(prof[('1gpu', 32)]['recs'], 0)
    ts2a, u2a, m2a, _ = gpu_ts(prof[('2gpu', 32)]['recs'], 0)
    ts2b, u2b, m2b, _ = gpu_ts(prof[('2gpu', 32)]['recs'], 1)
    for ax, one, two_a, two_b, ylabel in [
        (ax_u, u1, u2a, u2b, 'GPU util (%)'),
        (ax_m, m1, m2a, m2b, 'GPU mem (GiB)'),
    ]:
        ax.plot(ts1, one, 'b-', linewidth=1.8, label='1 GPU, GPU0')
        ax.plot(ts2a, two_a, 'r-', linewidth=1.8, label='2 GPU, GPU0')
        ax.plot(ts2b, two_b, 'r--', linewidth=1.3, label='2 GPU, GPU1')
        ax.set_xlabel('Time (s)')
        ax.set_ylabel(ylabel)
        ax.legend()
        ax.grid(alpha=0.3)
        ax.set_ylim(bottom=0)
    plt.tight_layout()
    plt.savefig('torch_profiling_allreduce.png', dpi=130, bbox_inches='tight')
    plt.show()
else:
    print('No 2-GPU profiling data to plot.')


## 5c. PCIe bandwidth & SM clock - batch-size sweep

Uses **pynvml** (`nvidia-ml-py`) to sample PCIe TX/RX bandwidth (KB/s) and SM
clock (MHz) from the driver every 250 ms while the PyTorch run executes in the
background.

`nvmlDeviceGetPcieThroughput` averages over 20 ms windows - values reflect
actual bytes flowing on the PCIe bus, not just link width.

- **1-GPU:** only tiny H2D copies (one batch ~ 100 KB) -> PCIe nearly idle between steps
- **2-GPU:** DDP allreduce adds ~`167k params x 4B = 654 KB` per step over PCIe

Also checks for **SM clock throttling** (thermal / power) by comparing current vs max MHz.

Runtime: same sweep as section 5 (~2-4 min). Run after section 5 so `prof` is in scope.


In [ ]:
import subprocess, threading, time, os
import numpy as np

try:
    import pynvml
    pynvml.nvmlInit()
except Exception:
    subprocess.run(['pip', 'install', '-q', 'nvidia-ml-py'], check=True)
    import pynvml
    pynvml.nvmlInit()

def _pcie_monitor(stop_evt, records, interval=0.25):
    n = pynvml.nvmlDeviceGetCount()
    handles = [pynvml.nvmlDeviceGetHandleByIndex(i) for i in range(n)]
    while not stop_evt.is_set():
        ts = time.time()
        for i, h in enumerate(handles):
            try:
                tx  = pynvml.nvmlDeviceGetPcieThroughput(h, pynvml.NVML_PCIE_UTIL_TX_BYTES)
                rx  = pynvml.nvmlDeviceGetPcieThroughput(h, pynvml.NVML_PCIE_UTIL_RX_BYTES)
                sm  = pynvml.nvmlDeviceGetClockInfo(h, pynvml.NVML_CLOCK_SM)
                mem = pynvml.nvmlDeviceGetClockInfo(h, pynvml.NVML_CLOCK_MEM)
                records.append(dict(ts=ts, gpu=i,
                                    tx_kbs=tx, rx_kbs=rx,
                                    sm_mhz=sm, mem_mhz=mem))
            except Exception:
                pass
        time.sleep(interval)

def run_pcie_prof(csv_path, B, steps=50, lr=0.001, np_=1):
    if np_ == 1:
        cmd = ['python', 'train_vit_torch.py', csv_path, str(steps), str(B), str(lr),
               '--device', 'cuda', '--log-path', 'torch_pcie_log.csv']
    else:
        cmd = ['torchrun', '--standalone', f'--nproc_per_node={np_}',
               '--master_port=29533', 'train_vit_torch.py',
               csv_path, str(steps), str(B), str(lr),
               '--device', 'cuda', '--log-path', 'torch_pcie_log.csv']
    stop = threading.Event()
    recs = []
    mon = threading.Thread(target=_pcie_monitor, args=(stop, recs), daemon=True)
    mon.start()
    t0 = time.time()
    subprocess.run(cmd, capture_output=True, text=True)
    elapsed = time.time() - t0
    stop.set()
    time.sleep(0.4)
    return dict(recs=recs, elapsed=elapsed)

CSV = os.environ.get('CSV', '')
assert CSV, 'Run the data cell first (section 3)'

n_gpus = int(subprocess.check_output(
    'nvidia-smi --query-gpu=name --format=csv,noheader | wc -l',
    shell=True, text=True).strip())

h0 = pynvml.nvmlDeviceGetHandleByIndex(0)
max_sm_mhz  = pynvml.nvmlDeviceGetMaxClockInfo(h0, pynvml.NVML_CLOCK_SM)
max_mem_mhz = pynvml.nvmlDeviceGetMaxClockInfo(h0, pynvml.NVML_CLOCK_MEM)
print(f'T4 max SM clock : {max_sm_mhz} MHz')
print(f'T4 max MEM clock: {max_mem_mhz} MHz')

pcie_prof = {}
BATCH_SIZES = [8, 16, 32, 64]

for B in BATCH_SIZES:
    print(f'  1-GPU  B={B:3d} / 50 steps ...', end=' ', flush=True)
    pcie_prof[('1gpu', B)] = run_pcie_prof(CSV, B, steps=50, np_=1)
    print(f"done  {pcie_prof[('1gpu', B)]['elapsed']:.1f}s")

if n_gpus >= 2:
    print('  2-GPU  B= 32 / 50 steps ...', end=' ', flush=True)
    pcie_prof[('2gpu', 32)] = run_pcie_prof(CSV, 32, steps=50, np_=2)
    print(f"done  {pcie_prof[('2gpu', 32)]['elapsed']:.1f}s")
else:
    print('Single-GPU session - 2-GPU PCIe sweep skipped.')

print('PCIe sweep complete.')


## 5d. GPU memory bandwidth occupancy

Estimates the fraction of T4's **300 GB/s** GDDR6 peak bandwidth in use,
derived from the memory-clock samples already collected in cell 5c.

Formula: `est_BW = mem_clock_MHz / max_mem_clock_MHz × 300 GB/s`

- **Memory clock at max** → full bandwidth available to the GPU
- **Memory clock throttled** → thermal or power cap is reducing effective bandwidth
- The [roofline (5f)](#roofline) shows this model is almost entirely memory-bound,
  so high memory-clock utilisation → near-peak performance

The T4's GDDR6 memory clock is less aggressively throttled than the SM clock
(which you saw in 5c); most variability here reflects power limits.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# T4 peak memory bandwidth — GDDR6, 256-bit bus
peak_bw_gbs = 300.0

BATCH_SIZES = [8, 16, 32, 64]
cmap = plt.cm.tab10(np.linspace(0, 0.8, len(BATCH_SIZES)))

def mem_bw_ts(recs, gpu_idx=0):
    g = [r for r in recs if r['gpu'] == gpu_idx]
    if not g:
        return [], []
    t0  = g[0]['ts']
    ts  = [r['ts'] - t0 for r in g]
    bw  = [r['mem_mhz'] / max_mem_mhz * peak_bw_gbs for r in g]
    return ts, bw

# Figure 1: time-series per batch size
fig1, axes = plt.subplots(1, len(BATCH_SIZES),
                          figsize=(5 * len(BATCH_SIZES), 5), sharey=True)
fig1.suptitle(
    f'Estimated GPU memory bandwidth — 50 steps, 1 GPU  '
    f'(GDDR6 peak = {peak_bw_gbs:.0f} GB/s)',
    fontsize=13, fontweight='bold')

for idx, (B, c) in enumerate(zip(BATCH_SIZES, cmap)):
    key = ('1gpu', B)
    ax  = axes[idx]
    if key not in pcie_prof:
        ax.set_visible(False)
        continue
    ts, bw = mem_bw_ts(pcie_prof[key]['recs'], 0)
    ax.plot(ts, bw, color=c, linewidth=1.3)
    ax.fill_between(ts, bw, alpha=0.18, color=c)
    ax.axhline(peak_bw_gbs, color='red', linewidth=1.0, linestyle='--',
               alpha=0.75, label=f'Peak {peak_bw_gbs:.0f} GB/s')
    ax.set_title(f'B={B}', fontsize=10)
    ax.set_xlabel('Time (s)', fontsize=9)
    if idx == 0:
        ax.set_ylabel('Est. bandwidth (GB/s)', fontsize=9)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    ax.set_ylim(0, peak_bw_gbs * 1.18)

plt.tight_layout()
plt.savefig('hbm_bw_timeseries.png', dpi=130, bbox_inches='tight')
plt.show()
print('Saved: hbm_bw_timeseries.png')

# Figure 2: peak vs avg BW summary bar
peak_bws, avg_bws = [], []
for B in BATCH_SIZES:
    key  = ('1gpu', B)
    recs = [r for r in pcie_prof.get(key, {}).get('recs', []) if r['gpu'] == 0]
    bws  = [r['mem_mhz'] / max_mem_mhz * peak_bw_gbs for r in recs] if recs else [0]
    peak_bws.append(max(bws))
    avg_bws.append(float(np.mean(bws)))

x      = np.arange(len(BATCH_SIZES))
xlbls  = [f'B={B}' for B in BATCH_SIZES]
w      = 0.35
fig2, ax2 = plt.subplots(figsize=(9, 5))
b1 = ax2.bar(x - w/2, peak_bws, w, label='Peak BW (GB/s)', color=cmap, edgecolor='white')
b2 = ax2.bar(x + w/2, avg_bws,  w, label='Avg BW (GB/s)',  color=cmap, alpha=0.55,
             edgecolor='black', linewidth=0.7)
ax2.bar_label(b1, fmt='%.0f', fontsize=9, padding=2)
ax2.bar_label(b2, fmt='%.0f', fontsize=9, padding=2)
ax2.axhline(peak_bw_gbs, color='red', linewidth=1.1, linestyle='--',
            alpha=0.7, label=f'GDDR6 ceiling {peak_bw_gbs:.0f} GB/s')
ax2.set_xticks(x); ax2.set_xticklabels(xlbls, fontsize=10)
ax2.set_ylabel('Bandwidth (GB/s)', fontsize=10)
ax2.set_title('Estimated GPU memory bandwidth by batch size', fontsize=12)
ax2.legend(fontsize=9)
ax2.grid(alpha=0.3, axis='y')
ax2.set_ylim(0, peak_bw_gbs * 1.22)

plt.tight_layout()
plt.savefig('hbm_bw_summary.png', dpi=130, bbox_inches='tight')
plt.show()
print('Saved: hbm_bw_summary.png')

print(f'T4 GDDR6 peak: {peak_bw_gbs:.0f} GB/s  |  max_mem_mhz = {max_mem_mhz} MHz')
for B, pk, av in zip(BATCH_SIZES, peak_bws, avg_bws):
    if pk > 0:
        print(f'  B={B:3d}: peak {pk:.1f} GB/s ({pk/peak_bw_gbs*100:.0f}%)'
              f'  avg {av:.1f} GB/s ({av/peak_bw_gbs*100:.0f}%)')
print('Note: bandwidth is estimated from memory-clock fraction × 300 GB/s.')
print('      For exact counters use Nsight Compute: ncu --metrics l1tex__t_bytes_pipe_lsu_mem_global_op_ld.sum ...')


## 5e. PCIe + NCCL overhead plots

Three figures from the PCIe sweep:

1. **Time-series** — PCIe TX/RX (MB/s) and SM clock (MHz) per batch size, 1-GPU.
   TX/RX should be near-zero between steps (only H2D batch copy).    SM clock below max → throttle.
2. **1-GPU vs 2-GPU PCIe comparison at B=32** — allreduce traffic appears as
   sharp periodic spikes on GPU0 and GPU1 simultaneously.
3. **Stacked bar: compute time vs NCCL time per step** — uses elapsed times from
   section 5. Shows what fraction of each step is idle waiting for allreduce.
   Effective NCCL bandwidth is computed as `data / overhead_time`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

BATCH_SIZES = [8, 16, 32, 64]
cmap = plt.cm.tab10(np.linspace(0, 0.8, len(BATCH_SIZES)))

def pcie_ts(recs, gpu_idx=0):
    g = [r for r in recs if r['gpu'] == gpu_idx]
    if not g:
        return [], [], [], []
    t0 = g[0]['ts']
    return (
        [r['ts'] - t0 for r in g],
        [r['tx_kbs'] / 1024 for r in g],   # MB/s
        [r['rx_kbs'] / 1024 for r in g],   # MB/s
        [r['sm_mhz'] for r in g],
    )

# Figure 1: PCIe TX/RX and SM clock per batch size (1-GPU)
fig1, axes = plt.subplots(2, len(BATCH_SIZES), figsize=(5 * len(BATCH_SIZES), 8))
fig1.suptitle('PCIe bandwidth & SM clock — 50 steps, 1 GPU', fontsize=13, fontweight='bold')

for idx, (B, c) in enumerate(zip(BATCH_SIZES, cmap)):
    key = ('1gpu', B)
    if key not in pcie_prof:
        axes[0][idx].set_visible(False)
        axes[1][idx].set_visible(False)
        continue
    ts, tx, rx, sm = pcie_ts(pcie_prof[key]['recs'], 0)

    ax_bw = axes[0][idx]
    ax_bw.plot(ts, tx, color='tab:blue', linewidth=1.2, label='TX (GPU->host)')
    ax_bw.plot(ts, rx, color='tab:orange', linewidth=1.2, label='RX (host->GPU)')
    ax_bw.fill_between(ts, tx, alpha=0.12, color='tab:blue')
    ax_bw.fill_between(ts, rx, alpha=0.12, color='tab:orange')
    ax_bw.set_title(f'B={B}  PCIe (MB/s)', fontsize=10)
    ax_bw.set_xlabel('Time (s)', fontsize=8)
    ax_bw.set_ylabel('MB/s', fontsize=8)
    ax_bw.legend(fontsize=7)
    ax_bw.grid(alpha=0.3)
    ax_bw.set_ylim(bottom=0)

    ax_clk = axes[1][idx]
    ax_clk.plot(ts, sm, color='tab:green', linewidth=1.2, label='SM clock')
    try:
        ax_clk.axhline(max_sm_mhz, color='r', linewidth=0.7, linestyle='--',
                       alpha=0.6, label=f'Max {max_sm_mhz} MHz')
    except NameError:
        pass
    ax_clk.set_title(f'B={B}  SM clock (MHz)', fontsize=10)
    ax_clk.set_xlabel('Time (s)', fontsize=8)
    ax_clk.set_ylabel('MHz', fontsize=8)
    ax_clk.legend(fontsize=7)
    ax_clk.grid(alpha=0.3)
    ax_clk.set_ylim(bottom=0)

plt.tight_layout()
plt.savefig('pcie_timeseries.png', dpi=130, bbox_inches='tight')
plt.show()
print('Saved: pcie_timeseries.png')

# Figure 2: 1-GPU vs 2-GPU PCIe — NCCL spikes visible
if ('2gpu', 32) in pcie_prof and ('1gpu', 32) in pcie_prof:
    fig2, (ax_tx, ax_rx) = plt.subplots(1, 2, figsize=(14, 5))
    fig2.suptitle('PCIe traffic: 1-GPU vs 2-GPU at B=32 — spikes = NCCL allreduce',
                  fontsize=12, fontweight='bold')

    ts1,  tx1,  rx1,  _ = pcie_ts(pcie_prof[('1gpu', 32)]['recs'], 0)
    ts2a, tx2a, rx2a, _ = pcie_ts(pcie_prof[('2gpu', 32)]['recs'], 0)
    ts2b, tx2b, rx2b, _ = pcie_ts(pcie_prof[('2gpu', 32)]['recs'], 1)

    for ax, y1, y2a, y2b, ylabel in [
        (ax_tx, tx1, tx2a, tx2b, 'PCIe TX MB/s  (GPU -> switch/peer)'),
        (ax_rx, rx1, rx2a, rx2b, 'PCIe RX MB/s  (switch/peer -> GPU)'),
    ]:
        ax.plot(ts1, y1, 'b-', linewidth=1.6, label='1-GPU  GPU0', alpha=0.9)
        ax.plot(ts2a, y2a, 'r-', linewidth=1.6, label='2-GPU  GPU0')
        ax.plot(ts2b, y2b, 'r--', linewidth=1.2, label='2-GPU  GPU1', alpha=0.8)
        ax.set_xlabel('Time (s)', fontsize=9)
        ax.set_ylabel(ylabel, fontsize=9)
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)
        ax.set_ylim(bottom=0)

    plt.tight_layout()
    plt.savefig('pcie_nccl_compare.png', dpi=130, bbox_inches='tight')
    plt.show()
    print('Saved: pcie_nccl_compare.png')
else:
    print('No 2-GPU PCIe data - run in T4x2 session.')

# Figure 3: NCCL overhead stacked bar (uses the prof dict from section 5)
have_timing = 'prof' in dir() and ('1gpu', 32) in prof and ('2gpu', 32) in prof
if have_timing:
    t1_total = prof[('1gpu', 32)]['elapsed']
    t2_total = prof[('2gpu', 32)]['elapsed']
    steps = 50
    nccl_s = max(0.0, t2_total - t1_total)
    ms_compute = t1_total / steps * 1000
    ms_nccl    = nccl_s / steps * 1000

    fig3, (ax_bar, ax_per) = plt.subplots(1, 2, figsize=(11, 5))
    fig3.suptitle('NCCL allreduce overhead vs compute time  (B=32, 50 steps)',
                  fontsize=12, fontweight='bold')

    ax_bar.bar(['1-GPU', '2-GPU'], [t1_total, t1_total],
               color='tab:blue', label='Compute (=1-GPU time)')
    ax_bar.bar(['2-GPU'], [nccl_s], bottom=[t1_total],
               color='tab:orange', label='NCCL overhead')
    ax_bar.set_ylabel('Total time (s)', fontsize=10)
    ax_bar.set_title('Total 50-step time', fontsize=10)
    ax_bar.legend(fontsize=9)
    ax_bar.grid(alpha=0.3, axis='y')

    ax_per.bar(['Compute/step', 'NCCL/step'], [ms_compute, ms_nccl],
               color=['tab:blue', 'tab:orange'])
    ax_per.set_ylabel('ms per step', fontsize=10)
    ax_per.set_title('Per-step breakdown', fontsize=10)
    ax_per.grid(alpha=0.3, axis='y')
    for i, v in enumerate([ms_compute, ms_nccl]):
        ax_per.text(i, v + 0.2, f'{v:.1f} ms', ha='center', fontsize=10)

    params = 167296
    data_kb = params * 4 / 1024
    if nccl_s > 0:
        eff_bw = params * 4 * steps / nccl_s / 1e9
        note = f'{data_kb:.0f} KB/step  |  effective NCCL BW ~ {eff_bw:.2f} GB/s  (PCIe 3.0 x16 peak = 16 GB/s)'
    else:
        note = f'{data_kb:.0f} KB/step  |  no measurable overhead (model too small or timing noise)'
    fig3.text(0.5, 0.01, note, ha='center', fontsize=9, style='italic')

    plt.tight_layout()
    plt.savefig('nccl_overhead.png', dpi=130, bbox_inches='tight')
    plt.show()
    if nccl_s > 0:
        frac = ms_nccl / (ms_compute + ms_nccl) * 100
        print(f'NCCL overhead: {ms_nccl:.1f} ms/step  ({frac:.0f}% of 2-GPU step time)')
        print(f'Effective NCCL BW: {eff_bw:.2f} GB/s  (PCIe 3.0 x16 peak = 16 GB/s)')
    print('Saved: nccl_overhead.png')
else:
    print('Run section 5 (GPU profiling sweep) with both 1-GPU and 2-GPU to see NCCL breakdown.')

print('PCIe / NCCL plots done.')


## 5f. Roofline analysis

**No training run needed** — purely analytical.

Plots each major operation on the T4 roofline:

- X-axis: **arithmetic intensity** = FLOP / byte (computed from model config)
- Y-axis: **performance ceiling** = min(peak FLOPS, bandwidth × AI)
- Ridge point: 8100 GFLOPS ÷ 300 GB/s = **27 FLOP/byte**

Key insight: with **D=64** this model is almost entirely **memory-bandwidth-bound**.
Even the largest matmuls (QKV proj, FC1, FC2) have AI ≈ 24–26, just below the ridge.
Adam step (AI ≈ 0.3) and embedding (AI ≈ 0.08) are deeply memory-bound.
To become compute-bound you would need D ≥ 256–512.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# T4 hardware limits
peak_gflops = 8100.0   # FP32 peak GFLOPS
peak_bw_gbs = 300.0    # HBM2 bandwidth GB/s
ridge_pt    = peak_gflops / peak_bw_gbs   # = 27 FLOP/byte

# Model config — matches compile-time defaults in train_vit.cu
B, T, D, H, L, C, V = 32, 784, 64, 4, 2, 10, 256
hd = D // H    # head_dim = 16
N  = 167296    # total trainable parameters

# (name, flop, bytes_accessed, group)
ops = [
    ('Embedding',
        B*T*D,
        (B*T*D + B*T*D + B*T*D) * 4,
        'memory'),
    ('LayerNorm',
        5 * B*T*D,
        (B*T*D + 2*D + B*T*D) * 4,
        'memory'),
    ('QKV proj  D->3D',
        2 * B*T * D * (3*D),
        (B*T*D + D*(3*D) + B*T*(3*D)) * 4,
        'matmul'),
    ('Attn QK  [T,hd]x[hd,T]',
        2 * B*H * T * hd * T,
        (B*H*T*hd + B*H*T*hd + B*H*T*T) * 4,
        'attention'),
    ('Attn softmax',
        5 * B*H * T*T,
        2 * B*H*T*T * 4,
        'memory'),
    ('Attn AV  [T,T]x[T,hd]',
        2 * B*H * T*T * hd,
        (B*H*T*T + B*H*T*hd + B*H*T*hd) * 4,
        'attention'),
    ('Attn out proj  D->D',
        2 * B*T * D*D,
        (B*T*D + D*D + B*T*D) * 4,
        'matmul'),
    ('MLP FC1  D->4D',
        2 * B*T * D * (4*D),
        (B*T*D + D*(4*D) + B*T*(4*D)) * 4,
        'matmul'),
    ('GELU',
        8 * B*T * (4*D),
        2 * B*T*(4*D) * 4,
        'memory'),
    ('MLP FC2  4D->D',
        2 * B*T * (4*D) * D,
        (B*T*(4*D) + (4*D)*D + B*T*D) * 4,
        'matmul'),
    ('Mean pool',
        B*T*D,
        (B*T*D + B*D) * 4,
        'memory'),
    ('Adam step',
        8 * N,
        7 * N * 4,
        'optimizer'),
]

color_map = {
    'memory':    '#4878D0',
    'matmul':    '#6ACC65',
    'attention': '#D65F5F',
    'optimizer': '#EE854A',
}

# Annotate only a few representative ops to avoid clutter
labeled = {
    'Embedding', 'Adam step',
    'Attn QK  [T,hd]x[hd,T]', 'QKV proj  D->3D', 'MLP FC1  D->4D',
}

fig, ax = plt.subplots(figsize=(13, 7))

ai_x = np.logspace(-2, 3, 500)
roof = np.minimum(peak_gflops, peak_bw_gbs * ai_x)
ax.loglog(ai_x, roof, 'k-', linewidth=2.2, label='Roofline (T4 FP32)')

ax.axvline(ridge_pt, color='k', linestyle='--', linewidth=0.9, alpha=0.45)
ax.text(ridge_pt * 1.1, 500,
        f'Ridge = {ridge_pt:.0f} FLOP/B', fontsize=8.5)
ax.text(0.013, peak_bw_gbs * 0.014,
        f'{peak_bw_gbs:.0f} GB/s HBM limit', fontsize=8, color='#555', rotation=41)
ax.text(300, peak_gflops * 1.12,
        f'{peak_gflops/1000:.1f} TFLOPS compute limit', fontsize=8, color='#555')

seen = set()
for name, flop, nbytes, grp in ops:
    ai   = flop / nbytes
    ceil = min(peak_gflops, peak_bw_gbs * ai)
    col  = color_map[grp]
    lbl  = grp.capitalize() if grp not in seen else None
    seen.add(grp)
    ax.scatter([ai], [ceil], color=col, s=95, zorder=5,
               edgecolors='white', linewidth=0.8, label=lbl)
    if name in labeled:
        ax.annotate(name, (ai, ceil),
                    textcoords='offset points', xytext=(6, 3),
                    fontsize=8, color=col)

ax.set_xlabel('Arithmetic Intensity  (FLOP / byte)', fontsize=11)
ax.set_ylabel('Performance ceiling  (GFLOPS)', fontsize=11)
ax.set_title(
    f'Roofline  —  T4  |  B={B} T={T} D={D} H={H} L={L}  '
    f'|  ridge = {ridge_pt:.0f} FLOP/byte  '
    '|  left of ridge = memory-bound,  right = compute-bound',
    fontsize=10)
ax.legend(fontsize=9, loc='upper left')
ax.grid(alpha=0.3, which='both')
ax.set_xlim(0.01, 800)
ax.set_ylim(1, peak_gflops * 3)

plt.tight_layout()
plt.savefig('roofline.png', dpi=150, bbox_inches='tight')
plt.show()

# Summary table
print(f'{"Operation":<28} {"AI (FLOP/B)":>12}   {"Bound":>10}   {"GFLOP ceiling":>14}')
print('-' * 72)
for name, flop, nbytes, _ in ops:
    ai   = flop / nbytes
    ceil = min(peak_gflops, peak_bw_gbs * ai)
    bound = 'COMPUTE' if ai > ridge_pt else 'MEMORY'
    print(f'{name:<28} {ai:>12.2f}   {bound:>10}   {ceil:>11.0f} GFLOPS')
print(f'\nRidge point: {ridge_pt:.1f} FLOP/byte')
print(f'T4 peak: {peak_gflops/1000:.1f} TFLOPS FP32  |  {peak_bw_gbs:.0f} GB/s HBM2')
print('Note: all ops in this D=64 model sit below or near the ridge -> bottleneck is HBM bandwidth.')
print('Saved: roofline.png')


## 6. Per-step timing breakdown

Reads the PyTorch training logs and shows how each step splits across the
pipeline stages timed with `torch.cuda.Event` in `train_vit_torch.py`:

| Column | What it measures |
|--------|------------------|
| `t_h2d_ms`  | Host->device transfer (pixels + labels) |
| `t_fwd_ms`  | Forward pass + cross-entropy loss |
| `t_bwd_ms`  | Backward pass - in DDP this also absorbs the gradient allreduce |
| `t_nccl_ms` | ~0 by construction: DDP overlaps the allreduce into `backward()`, so there is no separable phase to time |
| `t_adam_ms` | Adam optimizer step |

Times are sampled every 10th step. Plots:
1. **Stacked area over training** - how stage times evolve.
2. **Average breakdown bar** - mean time per stage.
3. **1-GPU vs 2-GPU comparison.**

Unlike the CUDA notebook there is **no fine-grained per-kernel breakdown**:
`nn.LayerNorm` / `nn.MultiheadAttention` / `nn.Linear` are opaque modules, so
CUDA events cannot be placed between their internal kernels.

The log copies appear after section 7; before that this cell falls back to the
profiling log from section 5.


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def load_log(path):
    return pd.read_csv(path) if os.path.exists(path) else None

logs = {}
for label, path in [('1 GPU', 'torch_1gpu_log.csv'),
                     ('2 GPU', 'torch_2gpu_log.csv')]:
    df = load_log(path)
    if df is not None:
        logs[label] = df
if not logs:
    for p in ['torch_profile_log.csv', 'training_log_torch.csv', 'training_log.csv']:
        df = load_log(p)
        if df is not None:
            logs['current'] = df
            break

COARSE_COLS   = ['t_h2d_ms', 't_fwd_ms', 't_bwd_ms', 't_nccl_ms', 't_adam_ms']
COARSE_LABELS = ['H2D', 'Forward', 'Backward', 'NCCL', 'Adam']
COARSE_COLORS = ['#4878D0', '#6ACC65', '#D65F5F', '#EE854A', '#956CB4']

if not logs:
    print('No log files found. Run section 5 (profiling) or section 7 (training) first.')
elif 't_fwd_ms' not in next(iter(logs.values())).columns:
    print('Logs have no timing columns - update train_vit_torch.py and re-run.')
else:
    nplots = len(logs)

    # 1. coarse stacked area
    fig1, axes = plt.subplots(1, nplots, figsize=(9 * nplots, 5), squeeze=False)
    fig1.suptitle('Per-step timing - coarse stages (stacked area)',
                  fontsize=13, fontweight='bold')
    for ax, (run_label, df) in zip(axes[0], logs.items()):
        bottom = np.zeros(len(df))
        for col, lbl, col_c in zip(COARSE_COLS, COARSE_LABELS, COARSE_COLORS):
            if col not in df.columns:
                continue
            vals = df[col].values
            ax.fill_between(df['step'], bottom, bottom + vals,
                            label=lbl, color=col_c, alpha=0.82)
            bottom += vals
        ax.set_xlabel('Step', fontsize=10)
        ax.set_ylabel('Time (ms)', fontsize=10)
        ax.set_title(run_label, fontsize=11)
        ax.legend(fontsize=9, loc='upper right')
        ax.grid(alpha=0.3, axis='y')
        ax.set_ylim(bottom=0)
    plt.tight_layout()
    plt.savefig('torch_timing_area.png', dpi=130, bbox_inches='tight')
    plt.show()
    print('Saved: torch_timing_area.png')

    # 2. coarse mean bars
    fig2, axes2 = plt.subplots(1, nplots, figsize=(6 * nplots, 5), squeeze=False)
    fig2.suptitle('Mean time per coarse stage (ms)', fontsize=13, fontweight='bold')
    for ax, (run_label, df) in zip(axes2[0], logs.items()):
        cols   = [c for c in COARSE_COLS if c in df.columns]
        labels = [COARSE_LABELS[COARSE_COLS.index(c)] for c in cols]
        colors = [COARSE_COLORS[COARSE_COLS.index(c)] for c in cols]
        means  = [df[c].mean() for c in cols]
        total  = sum(means)
        bars = ax.bar(labels, means, color=colors, edgecolor='white', linewidth=0.6)
        ax.bar_label(bars, fmt='%.2f ms', fontsize=9, padding=3)
        ax.set_ylabel('ms', fontsize=10)
        ax.set_title(f'{run_label}  |  {total:.1f} ms/step', fontsize=11)
        ax.grid(alpha=0.3, axis='y')
        ax.set_ylim(0, max(means) * 1.25 if means else 1)
        print(f'\n{run_label}:')
        for lbl, m in zip(labels, means):
            pct = f'  ({m / total * 100:5.1f}%)' if total else ''
            print(f'  {lbl:<10} {m:7.3f} ms{pct}')
        print(f'  {"total":<10} {total:7.3f} ms')
    plt.tight_layout()
    plt.savefig('torch_timing_bars.png', dpi=130, bbox_inches='tight')
    plt.show()
    print('\nSaved: torch_timing_bars.png')

    # 3. 1-GPU vs 2-GPU comparison
    if '1 GPU' in logs and '2 GPU' in logs:
        df1, df2 = logs['1 GPU'], logs['2 GPU']
        cols   = [c for c in COARSE_COLS if c in df1.columns and c in df2.columns]
        labels = [COARSE_LABELS[COARSE_COLS.index(c)] for c in cols]
        colors = [COARSE_COLORS[COARSE_COLS.index(c)] for c in cols]
        m1 = [df1[c].mean() for c in cols]
        m2 = [df2[c].mean() for c in cols]
        x = np.arange(len(labels)); w = 0.35
        fig3, ax3 = plt.subplots(figsize=(10, 5))
        b1 = ax3.bar(x - w/2, m1, w, label='1 GPU', color=colors, alpha=0.85,
                     edgecolor='white')
        b2 = ax3.bar(x + w/2, m2, w, label='2 GPU', color=colors, alpha=0.55,
                     edgecolor='black', linewidth=0.7)
        ax3.bar_label(b1, fmt='%.2f', fontsize=8, padding=2)
        ax3.bar_label(b2, fmt='%.2f', fontsize=8, padding=2)
        ax3.set_xticks(x); ax3.set_xticklabels(labels, fontsize=10)
        ax3.set_ylabel('Mean time (ms)', fontsize=10)
        ax3.set_title('1 GPU vs 2 GPU - coarse stage comparison', fontsize=11)
        ax3.legend(fontsize=10); ax3.grid(alpha=0.3, axis='y')
        plt.tight_layout()
        plt.savefig('torch_timing_1v2gpu.png', dpi=130, bbox_inches='tight')
        plt.show()
        print('Saved: torch_timing_1v2gpu.png')

    print('\nNote: PyTorch logs are coarse-only. Per-kernel timing is not')
    print('available - nn.* modules are opaque (see section 6 markdown).')
    print('t_nccl is ~0 by construction: DDP folds the allreduce into backward().')


## 7. Training runs - long experiments

Run these last, after sections 1-6. Each run is ~10-15 min.

- **1-GPU:** 7a -> 7b
- **2-GPU:** 7c -> 7d -> 7e


### 7a. Single-GPU run (Adam, 3000 steps)

Adam lr=1e-3, per-rank B=32. Writes `training_log_torch.csv`.
Run **7b** afterwards to rename the log.


In [ ]:
!python train_vit_torch.py "$CSV" 3000 32 0.001 \
  --device cuda --log-path training_log_torch.csv


### 7b. Save the 1-GPU log

Rename so the 2-GPU run can write a fresh `training_log_torch.csv`.


In [ ]:
import os, shutil
shutil.copy('training_log_torch.csv', 'torch_1gpu_log.csv')
print('Saved torch_1gpu_log.csv:', os.path.getsize('torch_1gpu_log.csv'), 'bytes')


### 7c. Two-GPU run (Adam, 3000 steps)

`torchrun` launches one rank per GPU. Per-rank B=32 -> global B=64.
PyTorch CE loss is a per-batch mean and DDP averages gradients across ranks,
so the update is the mean gradient over the global batch - the same semantics
as the CUDA run.


In [ ]:
!torchrun --standalone --nproc_per_node=2 --master_port=29541 \
  train_vit_torch.py "$CSV" 3000 32 0.001 \
  --device cuda --log-path training_log_torch.csv


### 7d. Save the 2-GPU log


In [ ]:
import os, shutil
shutil.copy('training_log_torch.csv', 'torch_2gpu_log.csv')
print('Saved torch_2gpu_log.csv:', os.path.getsize('torch_2gpu_log.csv'), 'bytes')


### 7e. Loss vs time: 1 GPU vs 2 GPU

X-axis is wall-clock time (seconds), not steps - this shows the actual speedup
from the second GPU.


In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

logs = {'PyTorch 1 GPU': 'torch_1gpu_log.csv',
        'PyTorch 2 GPU': 'torch_2gpu_log.csv'}
fig, (ax_loss, ax_acc) = plt.subplots(1, 2, figsize=(14, 5))

for label, path in logs.items():
    if not os.path.exists(path):
        print('Missing:', path)
        continue
    df = pd.read_csv(path)
    smooth = lambda s: s.rolling(window=10, min_periods=1).mean()
    line, = ax_loss.plot(df['elapsed_s'], smooth(df['loss']),
                         label=label, linewidth=2)
    ax_loss.plot(df['elapsed_s'], df['loss'], alpha=0.15, color=line.get_color())
    line, = ax_acc.plot(df['elapsed_s'], smooth(df['accuracy']),
                        label=label, linewidth=2)
    ax_acc.plot(df['elapsed_s'], df['accuracy'], alpha=0.15, color=line.get_color())

ax_loss.set_xlabel('Wall-clock time, seconds')
ax_loss.set_ylabel('Cross-entropy loss')
ax_loss.set_title('Loss vs time')
ax_loss.legend()
ax_loss.grid(alpha=0.3)
ax_acc.set_xlabel('Wall-clock time, seconds')
ax_acc.set_ylabel('Accuracy')
ax_acc.set_title('Accuracy vs time')
ax_acc.yaxis.set_major_formatter(ticker.PercentFormatter(xmax=1))
ax_acc.legend()
ax_acc.grid(alpha=0.3)
plt.suptitle('PyTorch 1 GPU vs 2 GPU - Adam lr=1e-3, per-rank B=32, 3000 steps',
             fontsize=13)
plt.tight_layout()
plt.savefig('torch_training_curves.png', dpi=150)
plt.show()


## 8. CUDA vs PyTorch comparison

Copy CUDA notebook logs into this session as:

- `cuda_1gpu_log.csv`
- `cuda_2gpu_log.csv`

The plot expects the same CSV dataset and the same run arguments. Compare
throughput and time-to-loss trends. Do not interpret point-by-point loss
differences as a kernel parity failure unless both implementations are fed the
same parameters and sampled batches.


In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

pairs = [
    ('1 GPU', 'cuda_1gpu_log.csv', 'torch_1gpu_log.csv'),
    ('2 GPU', 'cuda_2gpu_log.csv', 'torch_2gpu_log.csv'),
]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for gpu_label, cuda_path, torch_path in pairs:
    for impl, path, style in [('CUDA', cuda_path, '-'), ('PyTorch', torch_path, '--')]:
        if not os.path.exists(path):
            print('Missing optional comparison log:', path)
            continue
        df = pd.read_csv(path)
        axes[0].plot(df['elapsed_s'], df['loss'].rolling(10, min_periods=1).mean(),
                     linestyle=style, linewidth=2, label=f'{impl} {gpu_label}')
        axes[1].plot(df['elapsed_s'],
                     df['accuracy'].rolling(10, min_periods=1).mean(),
                     linestyle=style, linewidth=2, label=f'{impl} {gpu_label}')

axes[0].set_xlabel('Wall-clock time, seconds')
axes[0].set_ylabel('Cross-entropy loss')
axes[0].set_title('Training loss')
axes[1].set_xlabel('Wall-clock time, seconds')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Training accuracy')
for ax in axes:
    ax.grid(alpha=0.3)
    ax.legend()
plt.suptitle('CUDA vs PyTorch logs with matching run settings', fontsize=13)
plt.tight_layout()
plt.show()


## Comparison notes

- The CUDA notebook and this notebook use the same data locator and fallback
  materialization to the Kaggle CSV format.
- The reported global batch is not the same between 1 GPU and 2 GPU runs:
  both notebooks use per-rank B=32, so two GPUs use global B=64.
- The PyTorch code uses standard `nn.Module`, `DataLoader`, Adam, and DDP
  rather than copying CUDA memory layout and kernel boundaries.
- PyTorch attention is pinned with `sdpa_kernel(SDPBackend.MATH)` for these
  comparisons.
- For a strict numerical forward/backward parity test, add a shared parameter
  fixture and a shared batch-index fixture to both implementations first.
- Logs carry coarse per-stage CUDA-event timings (H2D/fwd/bwd/nccl/adam). Fine-grained per-kernel timing is CUDA-notebook-only - see section 6.
